## EDA

In [ ]:
import pandas as pd
import numpy as np

#aqui solo se cargan las columnas que pide el profesor porque luego me quedaba sin espacio
cols_necesarias = [
    # Numéricas
    'loan_amnt', 'int_rate', 'annual_inc', 'dti',
    'fico_range_high', 'emp_length',
    # Categóricas
    'purpose', 'home_ownership', 'addr_state', 'verification_status',
    # Target
    'loan_status',
    # Extras (por si acaso)
    'grade', 'sub_grade', 'term', 'installment',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc', 'mort_acc', 'pub_rec_bankruptcies',
    'funded_amnt', 'issue_d'
]

df = pd.read_csv('data/lending_club/accepted_2007_to_2018Q4.csv.gz',
                 compression='gzip',
                 usecols=cols_necesarias,
                 low_memory=False)

print(f"  Filas:    {df.shape[0]:,}")
print(f"  Columnas: {df.shape[1]:,}")

In [ ]:
#Mostrar las primeras y últimas filas (head(), tail())
print("\n=== PRIMERAS 5 FILAS ===")
display(df.head())

print("\n=== ÚLTIMAS 5 FILAS ===")
display(df.tail())

#Usar info() y describe() para obtener un resumen de tipos de datos
df.info(verbose=True, show_counts=True)
display(df.describe())

#Reportar la dimensión del dataset (número de filas y columnas).
print(f"Dimensión del dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")

#target
df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)

print("Valores únicos de loan_status:")
print(df["loan_status"].value_counts())

print("\nDistribución del target (default):")
conteo = df["default"].value_counts()
porcentaje = df["default"].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({'Conteo': conteo, 'Porcentaje %': porcentaje}))

## Análisis unidimensional

### Variables numéricas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

vars_numericas = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_high', 'emp_length']
display(df[vars_numericas].describe())

if df['emp_length'].dtype == object:
    df['emp_length'] = df['emp_length'].str.extract(r'(\d+)').astype(float)

#VALORES ATÍPICOS
print("OUTLIERS POR MÉTODO IQR \n")

for col in vars_numericas:
    serie = df[col].dropna()
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    outliers = serie[(serie < limite_inf) | (serie > limite_sup)]
    pct = round(len(outliers) / len(serie) * 100, 2)
    print(f"{col:20s} | Límite inf: {limite_inf:>12.2f} | Límite sup: {limite_sup:>12.2f} | Outliers: {len(outliers):>7,} ({pct}%)")

#HISTOGRAMAS
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(vars_numericas):
    axes[i].hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='white')
    axes[i].set_title(f'Distribución de {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Histogramas — Variables Numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

#DIAGRAMAS DE CAJA
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(vars_numericas):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'Boxplot de {col}', fontsize=12)
    axes[i].set_ylabel(col)

plt.suptitle('Boxplots — Variables Numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

#NORMALIDAD
print("ASIMETRÍA\n")
print(f"{'Variable':20s} {'Asimetría':>12} {'Interpretación':>25}")
print("-" * 60)

for col in vars_numericas:
    skew = df[col].dropna().skew()
    if abs(skew) < 0.5:
        interp = "Normal"
    elif abs(skew) < 1:
        interp = "Moderada"
    else:
        interp = "Alta"
    print(f"{col:20s} {skew:>12.4f} {interp:>25}")
#segun los resultados de asimetria, annual_inc, dti y fico_range_high necesitan transformaciones

#TASA DE VALORES FALTANTES
nulos = df[vars_numericas].isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

tabla_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje %': pct_nulos,
    'Tratamiento': ''
})

# Aplicar umbral (> 30%)
for col in tabla_nulos.index:
    pct = tabla_nulos.loc[col, 'Porcentaje %']
    if pct > 30:
        tabla_nulos.loc[col, 'Tratamiento'] = 'Eliminación'
    elif pct > 5:
        tabla_nulos.loc[col, 'Tratamiento'] = 'Imputación avanzada'
    elif pct > 0:
        tabla_nulos.loc[col, 'Tratamiento'] = 'Imputar con mediana'
    else:
        tabla_nulos.loc[col, 'Tratamiento'] = 'Sin nulos'

display(tabla_nulos)

### Variables categóricas

In [ ]:
#Frecuencia absoluta y relativa
vars_categoricas = ['purpose', 'home_ownership', 'addr_state', 'verification_status']

for col in vars_categoricas:
    print(f"\n{'='*55}")
    print(f"  VARIABLE: {col.upper()}")
    print(f"{'='*55}")

    freq_abs = df[col].value_counts()
    freq_rel = df[col].value_counts(normalize=True).mul(100).round(2)

    tabla = pd.DataFrame({
        'Frecuencia Absoluta': freq_abs,
        'Frecuencia Relativa %': freq_rel
    })
    display(tabla)

#Identificar categorías con muy baja frecuencia que podrían agruparse en una categoría “Otros”. (<1%)
print("CATEGORÍAS CON FRECUENCIA BAJA\n")

for col in vars_categoricas:
    freq_rel = df[col].value_counts(normalize=True).mul(100).round(2)
    raras = freq_rel[freq_rel < 1]
    if len(raras) > 0:
        print(f"\n{col}:")
        for cat, pct in raras.items():
            print(f"  '{cat}': {pct}%")
    else:
        print(f"\n{col}: No hay categorías con frecuencia < 1%")

#Generar gráficos de barras horizontales o verticales para visualizar la distribución.
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(vars_categoricas):
    freq = df[col].value_counts()
    axes[i].barh(freq.index, freq.values, color='steelblue')
    axes[i].set_title(f'Distribución de {col}', fontsize=12)
    axes[i].set_xlabel('Frecuencia')
    for j, v in enumerate(freq.values):
        axes[i].text(v, j, f' {v:,}', va='center', fontsize=8)

plt.suptitle('Variables Categóricas', fontsize=14)
plt.tight_layout()
plt.show()

#Detectar si existen categorías con significado redundante o que puedan unificarse.

#Calcular la tasa de valores faltantes y considerar su tratamiento (moda, categoría “Desconocido”, o eliminación si es muy alta).
print("VALORES FALTANTES — VARIABLES CATEGÓRICAS\n")

for col in vars_categoricas:
    nulos = df[col].isnull().sum()
    pct = round(nulos / len(df) * 100, 2)

    if pct > 30:
        decision = 'Eliminar columna'
    elif pct > 5:
        decision = 'Imputar con categoría Desconocido'
    elif pct > 0:
        decision = 'Imputar con moda'
    else:
        decision = 'Sin nulos'

    print(f"{col:25s} | Nulos: {nulos:>7,} | {pct:>5}% | {decision}")

### Variable objetivo

In [ ]:
#Calcular la distribución de clases (0 = Fully Paid, 1 = Charged Off) en valores absolutos y porcentajes.
df_eda = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df_eda['default'] = (df_eda['loan_status'] == 'Charged Off').astype(int)

conteo = df_eda['default'].value_counts().sort_index()
porcentaje = df_eda['default'].value_counts(normalize=True).sort_index() * 100

dist_target = pd.DataFrame({
    'Clase': ['0', '1'],
    'Conteo': conteo.values,
    'Porcentaje (%)': porcentaje.values.round(2)
})
dist_target.index = ['Fully Paid', 'Charged Off']

print("=" * 55)
print("   DISTRIBUCIÓN DE LA VARIABLE OBJETIVO — default")
print("=" * 55)
print(dist_target.to_string())
print("=" * 55)
print(f"   Total de registros válidos: {len(df_eda):,}")
print(f"   Desbalance (mayoría/minoría): "
      f"{conteo[0]/conteo[1]:.2f}:1")
print("=" * 55)

#Visualizar con un gráfico de barras o pastel.
PALETTE = {'Fully Paid': '#2ecc71', 'Charged Off': '#e74c3c'}

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    ['Fully Paid\n(0)', 'Charged Off\n(1)'],
    conteo.values,
    color=[PALETTE['Fully Paid'], PALETTE['Charged Off']],
    edgecolor='white', linewidth=1.2, width=0.5
)
for bar, cnt, pct in zip(bars, conteo.values, porcentaje.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + conteo.max() * 0.01,
        f'{cnt:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('Distribución de la Variable Objetivo',
             fontsize=10, fontweight='bold')
ax.set_ylabel('Número de préstamos', fontsize=10)
ax.set_ylim(0, conteo.max() * 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## Análisis bidimensional

### Relación numérica ↔ default

In [ ]:
#Calcular la diferencia de medias entre clases y realizar una prueba t de Student o prueba de Mann-Whitney
#Calcular la correlación punto-biserial entre cada variable numérica y la variable binaria default.
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

vars_num = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
            'fico_range_high', 'emp_length']

clase0 = df_eda[df_eda['default'] == 0]
clase1 = df_eda[df_eda['default'] == 1]

print("  DIFERENCIA DE MEDIAS, PRUEBA MANN-WHITNEY Y CORRELACIÓN PUNTO-BISERIAL \n")

print(f"{'Variable':<20} {'Media_0':>10} {'Media_1':>10} {'Δ Media':>10} "
      f"{'p-value':>12} {'Significancia':>6} {'r_pb':>8}")
print("-" * 90)

resultados = {}
for var in vars_num:
    s0 = clase0[var].dropna()
    s1 = clase1[var].dropna()

    media0 = s0.mean()
    media1 = s1.mean()
    delta  = media1 - media0

    # Mann-Whitney
    stat, pval = stats.mannwhitneyu(s0, s1, alternative='two-sided')

    # Correlación punto-biserial
    serie_var  = df_eda[var].dropna()
    serie_def  = df_eda.loc[serie_var.index, 'default']
    rpb, _     = stats.pointbiserialr(serie_def, serie_var)

    sig = 'Muy alta' if pval < 0.001 else ('Alta' if pval < 0.01 else
          ('Regular'  if pval < 0.05  else 'NS'))

    resultados[var] = dict(media0=media0, media1=media1,
                           delta=delta, pval=pval, sig=sig, rpb=rpb)

    print(f"{var:<20} {media0:>10.2f} {media1:>10.2f} {delta:>10.2f} "
          f"{pval:>12.2e} {sig:>6} {rpb:>8.4f}")

#Generar boxplots comparativos por clase (default = 0 vs 1) para cada variable numérica
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
PALETTE = {0: '#2ecc71', 1: '#e74c3c'}
LABELS  = {0: 'Fully Paid (0)', 1: 'Charged Off (1)'}

for i, var in enumerate(vars_num):
    ax = axes[i]
    data_plot = [clase0[var].dropna(), clase1[var].dropna()]

    bp = ax.boxplot(data_plot, patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', markersize=2,
                                   alpha=0.3, linestyle='none'))

    for patch, color in zip(bp['boxes'], PALETTE.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    r   = resultados[var]
    sig = r['sig']
    ax.set_title(f"{var}\np={r['pval']:.2e} {sig}  |  r_pb={r['rpb']:.3f}",
                 fontsize=10)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Fully Paid (0)', 'Charged Off (1)'], fontsize=9)
    ax.set_ylabel(var, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Boxplots - Variables Numéricas por Clase',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Relación categórica ↔ default

In [ ]:
from scipy.stats import chi2_contingency
vars_cat = ['purpose', 'home_ownership', 'addr_state', 'verification_status']

#PRUEBA DE CHI-CUADRADO
print("  PRUEBA CHI-CUADRADO: VARIABLES CATEGÓRICAS vs DEFAULT \n")
print(f"{'Variable':<22} {'Chi2':>12} {'p-value':>12} {'Significancia':>6} {'V Cramer':>10}")
print("-" * 65)

resultados_cat = {}
for var in vars_cat:
    tabla = pd.crosstab(df_eda[var], df_eda['default'])
    chi2, pval, dof, _ = chi2_contingency(tabla)

    # V de Cramer (magnitud del efecto)
    n = tabla.values.sum()
    k = min(tabla.shape) - 1
    v_cramer = np.sqrt(chi2 / (n * k))

    sig = 'Muy Alta' if pval < 0.001 else ('Alta' if pval < 0.01 else
          ('Regular'  if pval < 0.05  else 'NS'))

    resultados_cat[var] = dict(chi2=chi2, pval=pval, sig=sig,
                                v_cramer=v_cramer, tabla=tabla)

    print(f"{var:<22} {chi2:>12.2f} {pval:>12.2e} {sig:>6} {v_cramer:>10.4f}")

#TABLAS DE CONTINGENCIA
print("\n TABLAS DE CONTINGENCIA \n")
for var in vars_cat:
    tabla = resultados_cat[var]['tabla'].copy()
    tabla.columns = ['Fully Paid (0)', 'Charged Off (1)']
    tabla['Total']          = tabla.sum(axis=1)
    tabla['Tasa Default (%)'] = (tabla['Charged Off (1)'] /
                                  tabla['Total'] * 100).round(2)
    tabla = tabla.sort_values('Tasa Default (%)', ascending=False)

    print(f" \n  {var.upper()}")
    print(f"  Chi2={resultados_cat[var]['chi2']:.2f}  "
          f"p={resultados_cat[var]['pval']:.2e}  "
          f"{resultados_cat[var]['sig']}  "
          f"V={resultados_cat[var]['v_cramer']:.4f}")
    print(f"{'─'*65}")
    print(tabla.to_string())

#GRÁFICOS DE BARRAS APILADAS
for var in vars_cat:
    tabla = resultados_cat[var]['tabla'].copy()
    tabla_pct = tabla.div(tabla.sum(axis=1), axis=0) * 100
    tabla_pct.columns = ['Fully Paid (0)', 'Charged Off (1)']
    tabla_pct = tabla_pct.sort_values('Charged Off (1)', ascending=False)

    if len(tabla_pct) > 15:
        tabla_pct = tabla_pct.head(15)
        titulo_extra = ' (Top 15)'
    else:
        titulo_extra = ''

    fig, ax = plt.subplots(figsize=(max(8, len(tabla_pct) * 0.7), 5))

    tabla_pct[['Fully Paid (0)', 'Charged Off (1)']].plot(
        kind='bar', stacked=True, ax=ax,
        color=['#2ecc71', '#e74c3c'],
        edgecolor='white', linewidth=0.5
    )

    tasa_global = df_eda['default'].mean() * 100
    ax.axhline(tasa_global, color='black', linestyle='--',
               linewidth=1.2, label=f'Tasa global ({tasa_global:.1f}%)')

    ax.set_title(
        f'{var}{titulo_extra}\n'
    )

    ax.set_xlabel(var, fontsize=10)
    ax.set_ylabel('Proporción (%)', fontsize=10)
    ax.set_ylim(0, 110)
    ax.legend(loc='upper right', fontsize=9, frameon=False)
    ax.tick_params(axis='x', rotation=45)
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.show()

### Multicolinealidad

In [ ]:
import seaborn as sns
#CALCULAR MATRIZ DE CORRELACIONES
vars_num = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
            'fico_range_high', 'emp_length']

corr_matrix = df_eda[vars_num].corr(method='pearson')

print("  MATRIZ DE CORRELACIÓN DE PEARSON — VARIABLES NUMÉRICAS \n")
print(corr_matrix.round(4).to_string())

#HEAT MAP
fig, ax = plt.subplots(figsize=(8, 6))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
mask_lower = np.tril(np.ones_like(corr_matrix, dtype=bool), k=-1)

sns.heatmap(
    corr_matrix,
    annot=True, fmt='.3f',
    cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    ax=ax,
    cbar_kws={'label': 'Correlación de Pearson', 'shrink': 0.8}
)

for i in range(len(vars_num)):
    for j in range(len(vars_num)):
        if i != j and abs(corr_matrix.iloc[i, j]) > 0.7:
            ax.add_patch(plt.Rectangle((j, i), 1, 1,
                         fill=False, edgecolor='black',
                         linewidth=2.5, zorder=3))

ax.set_title('Matriz de Correlación de Pearson \n',
             fontsize=11, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

#ASOCIACÓN
vars_cat = ['purpose', 'home_ownership', 'addr_state', 'verification_status']

def v_cramer(x, y):
    tabla = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(tabla)
    n = tabla.values.sum()
    k = min(tabla.shape) - 1
    return np.sqrt(chi2 / (n * k))

n_cat = len(vars_cat)
v_matrix = pd.DataFrame(np.zeros((n_cat, n_cat)),
                         index=vars_cat, columns=vars_cat)

for i, v1 in enumerate(vars_cat):
    for j, v2 in enumerate(vars_cat):
        if i == j:
            v_matrix.loc[v1, v2] = 1.0
        elif i < j:
            v = v_cramer(df_eda[v1], df_eda[v2])
            v_matrix.loc[v1, v2] = v
            v_matrix.loc[v2, v1] = v

print("  MATRIZ V DE CRAMER — VARIABLES CATEGÓRICAS \n")
print(v_matrix.round(4).to_string())

print("\n  PARES CON ASOCIACIÓN NOTABLE (V > 0.1):")
for i in range(n_cat):
    for j in range(i+1, n_cat):
        v = v_matrix.iloc[i, j]
        if v > 0.1:
            print(f"  {vars_cat[i]} — {vars_cat[j]:<25} {v:>8.4f} ")

### KDE

In [ ]:
#GRÁFICOS DE DENSIDAD (KDE)
vars_kde = ['int_rate', 'fico_range_high', 'dti', 'annual_inc',
            'loan_amnt', 'emp_length']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

COLORES = {0: '#2ecc71', 1: '#e74c3c'}
LABELS  = {0: 'Fully Paid (0)', 1: 'Charged Off (1)'}

for i, var in enumerate(vars_kde):
    ax = axes[i]

    for clase in [0, 1]:
        datos = df_eda[df_eda['default'] == clase][var].dropna()

        # KDE
        datos.plot.kde(ax=ax, color=COLORES[clase],
                       linewidth=2, label=LABELS[clase])

        # Área bajo la curva
        kde = stats.gaussian_kde(datos.sample(min(len(datos), 50_000),
                                              random_state=42))
        x_range = np.linspace(datos.quantile(0.01),
                              datos.quantile(0.99), 300)
        ax.fill_between(x_range, kde(x_range),
                        alpha=0.15, color=COLORES[clase])

        # Línea de mediana
        mediana = datos.median()
        ax.axvline(mediana, color=COLORES[clase],
                   linestyle='--', linewidth=1.2, alpha=0.8)

    ax.set_title(var, fontsize=11, fontweight='bold')
    ax.set_xlabel(var, fontsize=9)
    ax.set_ylabel('Densidad', fontsize=9)
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)

    xmin = df_eda[var].quantile(0.01)
    xmax = df_eda[var].quantile(0.99)
    ax.set_xlim(xmin, xmax)

fig.suptitle('KDE por Clase de Default — Variables Numéricas\n'
             '(líneas punteadas = mediana por clase)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Valores faltantes

In [ ]:
import missingno as msno

vars_todas = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
              'fico_range_high', 'emp_length',
              'purpose', 'home_ownership', 'addr_state',
              'verification_status', 'default']

df_miss = df_eda[vars_todas].copy()

#Reportar, para cada columna, el número y porcentaje de valores nulos.
nulos      = df_miss.isnull().sum()
porcentaje = (nulos / len(df_miss) * 100).round(4)
dtype_col  = df_miss.dtypes

tabla_nulos = pd.DataFrame({
    'Tipo'       : dtype_col,
    'Nulos'      : nulos,
    'Porcentaje' : porcentaje,
}).sort_values('Porcentaje', ascending=False)

print("  VALORES FALTANTES POR VARIABLE \n")
print(tabla_nulos.to_string())

#Visualizar el patrón de missing values
muestra = df_miss.sample(min(10_000, len(df_miss)), random_state=42)

# 2b. Heatmap de correlación de nulos
fig, ax = plt.subplots(figsize=(9, 6))
msno.heatmap(muestra, ax=ax, fontsize=10, cmap='coolwarm')
ax.set_title('Correlación entre patrones de nulos\n',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


#Analizar valores faltantes
print("  PRUEBA MCAR \n")
print(f"{'Variable':<22} {'% null_default0':>16} {'% null_default1':>16} "
      f"{'p-value':>10} {'Significancia':>6}")
print("-" * 65)

for var in vars_todas:
    if var == 'default':
        continue
    n_nulos = df_miss[var].isnull().sum()
    if n_nulos == 0:
        print(f"{var:<22} {'—':>16} {'—':>16} {'—':>10} {'Sin nulos':>10}")
        continue

    # Crear indicador binario de missing
    indicador = df_miss[var].isnull().astype(int)
    tabla_chi  = pd.crosstab(indicador, df_miss['default'])

    if tabla_chi.shape == (2, 2):
        chi2, pval, _, _ = chi2_contingency(tabla_chi)
        sig = 'Muy Alta' if pval < 0.001 else ('Alta' if pval < 0.01 else
              ('Regular'  if pval < 0.05  else 'NS'))

        # Tasa de nulos por clase
        pct0 = (df_miss[df_miss['default']==0][var].isnull().mean()*100)
        pct1 = (df_miss[df_miss['default']==1][var].isnull().mean()*100)
        print(f"{var:<22} {pct0:>15.2f}% {pct1:>15.2f}% "
              f"{pval:>10.2e} {sig:>6}")
    else:
        print(f"{var:<22} {'—':>16} {'—':>16} {'solo 1 clase':>10}")